kernel : Python (Pixi)

In [1]:
import anndata
import numpy as np
import pandas as pd
import os
import scanpy as sc
from anndata import AnnData
from scipy import sparse
from scipy.sparse import csr_matrix
from pipeline.utils import plot
from pipeline.utils.deseq2 import pseudobulk_deseq2, pseudobulk_deseq2_comp
from pipeline.utils.env import find_env_dir
from pipeline.utils.find_reference_gene import find_pseudobulk_reference_genes

anndata.settings.allow_write_nullable_strings = True

series_name = "Mace"
clustered_data_location = find_env_dir("CLUSTERED_DATA")

mace_adata = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))

In [ ]:
cell_marker_genes = {
    "OPC": ["Pdgfra", "Megf11", "Vcan", "Has2", "Col9a1", ],
    "COP": ["Gpr17", "Tcf7l2", ],
    "Oligodendrocyte": ["Mbp", "Mog", "Mag", "Cnp", "Mobp", "Fa2h",
                        "Ppp1r14a", "Ermn", "Gpr37", "Myrf", ],
    "Neuron": ["Stmn2", "Rbfox3", "Eno2", "Syt1", "Grin1", "Tmem130", "Syp"],
    "Microglia": ["Tmem119", "Siglech", "Gpr34", ],
    "Tcell": ["Cd3d", "Cd3e", "Cd3g", "Trac", "S100a4", "Cd2", "Cd5", ],
    "NKcell": ["Nkg7", "Klrd1", "Prf1", "Ncr1", ],
    "Granulocyte": ["Cxcr2", "S100a8", "S100a9", "Hdc", ],
    "DC": ["Itgax", "Clec10a", "Xcr1", "Clec9a", ],
    "Macrophage": ["Ms4a7", "Lgals3", "Tgfbi", "Lyz2", "Ccl9", ],
    "Endothelial": ["Cldn5", "Vwf", "Cdh5", "Clec14a", "Erg", "Itm2a", ],
    "Pericyte": ["Rgs5", "Cd248", "Kcnj8", ],
    "Astrocyte": ["Aqp4", "Aldh1l1", "Aldh1a1", "Agt", "Atp13a4", "Bmpr1b", "Rfx4", "Slc4a4", ],
    "Ependymal": ["Foxj1", "Dynlrb2", "Dnah11", "Pifo", ],
    "VLMC": ["Col1a2", "Col3a1", ],
}

plot.plot_dotplot(mace_adata, series_name, cell_marker_genes, group = "leiden")

In [62]:
leiden_to_celltype = {
    "0": "Macrophage",
    "1": "Macrophage",
    "2": "Microglia",
    "3": "Macrophage",
    "4": "Astrocyte",
    "5": "Tcell",
    "6": "Neuron",
    "7": "Neuron",
    "8": "Macrophage",
    "9": "Microglia",
    "10": "Oligodendrocyte",
    "11": "Neuron",
    "12": "Microglia",
    "13": "Microglia",
    "14": "OPC",
    "15": "Neuron",
    "16": "Oligodendrocyte",
    "17": "Neuron",
    "18": "Granulocyte",
    "19": "Neuron",
    "20": "Astrocyte",
    "21": "OPC",
    "22": "Oligodendrocyte",
    "23": "Neuron",
    "24": "Neuron",
    "25": "Astrocyte",
    "26": "Microglia",
    "27": "Neuron",
    "28": "Neuron",
    "29": "Neuron",
    "30": "Oligodendrocyte",
    "31": "Oligodendrocyte",
    "32": "Neuron",
    "33": "Oligodendrocyte",
    "34": "Macrophage",
    "35": "Macrophage",
    "36": "Neuron",
    "37": "Pericyte",
    "38": "Neuron",
    "39": "COP",
    "40": "Neuron",
    "41": "Tcell",
    "42": "Tcell",
    "43": "Ependymal",
    "44": "Oligodendrocyte",
    "45": "Neuron",
    "46": "Oligodendrocyte",
    "47": "Neuron",
    "48": "Neuron",
    "49": "Oligodendrocyte",
    "50": "VLMC",
    "51": "Neuron",
    "52": "Neuron",
    "53": "DC",
    "54": "Neuron",
    "55": "Neuron",
    "56": "Tcell",
    "57": "Neuron",
    "58": "Tcell",
    "59": "Neuron",
    "60": "Neuron",
    "61": "Oligodendrocyte",
    "62": "Neuron",
    "63": "Neuron", 
    "64": "Neuron",
    "65": "Neuron",
    "66": "Neuron",
    "67": "Endothelial",
    "68": "Neuron",
    "69": "Astrocyte",
    "70": "Neuron",
    "71": "Astrocyte",
    "72": "NKcell",
    "73": "Oligodendrocyte",
    "74": "Astrocyte",
    "75": "Neuron",
    "76": "Microglia",
}

del mace_adata.obsp

leiden_str = mace_adata.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
mace_adata.obs["cell"] = mapped

mace_adata = mace_adata[mace_adata.obs["cell"] != "Doublet"].copy()

In [63]:
mace_adata.write_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))

In [65]:
plot.plot_umap(mace_adata, series_name, has_celltype=True)

In [64]:
h5ad_matrix_location = find_env_dir("H5AD_MATRIX")

opc = mace_adata[
    (mace_adata.obs["cell"] == "OPC") |
    (mace_adata.obs["cell"] == "COP")
]
opc.write_h5ad(os.path.join(h5ad_matrix_location, series_name + "_opc.h5ad"))

oligodendrocyte = mace_adata[mace_adata.obs["cell"] == "Oligodendrocyte"]
oligodendrocyte.write_h5ad(os.path.join(h5ad_matrix_location, series_name + "_oligodendrocyte.h5ad"))

oligo = mace_adata[
    (mace_adata.obs["cell"] == "OPC") |
    (mace_adata.obs["cell"] == "COP") | 
    (mace_adata.obs["cell"] == "Oligodendrocyte")
]
oligo.write_h5ad(os.path.join(h5ad_matrix_location, series_name + "_oligo.h5ad"))